# UNet++ for Remote Sensing Change Detection

This notebook implements **UNet++** (Nested UNet) for pixel-level change detection using satellite/aerial imagery. UNet++ improves on the standard U-Net by introducing **nested skip connections** with dense convolution blocks, which reduce the semantic gap between encoder and decoder feature maps.

## Key Differences: UNet vs UNet++
- **UNet**: Simple skip connections directly concatenate encoder features to decoder features of the same resolution.
- **UNet++**: Each skip connection passes through a series of dense convolutional blocks, progressively refining features before concatenation. This enables better gradient flow and captures multi-scale features more effectively.

## Model Architecture
- **Depth**: 5 levels (encoder/decoder stages with progressive down/up-sampling)
- **Base filters**: 32, doubling at each level until 512 at the bottleneck
- **Input**: 6-channel image pair (3-channel from each time step, concatenated)
- **Output**: Single-channel binary change mask with sigmoid activation
- **Loss**: Combined Binary Cross-Entropy + Dice Loss
- **Optimizer**: Adam

## 1. Setup and Imports

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

## 2. Dataset Download

Dataset: **Change Detection Dataset** (VHR satellite imagery) containing image pairs (A and B) with binary change labels.

Source: https://drive.google.com/file/d/1GX656JqqOyBi_Ef0w65kDGVto-nHrNs9

In [ ]:
DATA_DIR = 'vhrdata'

if not os.path.exists(DATA_DIR):
    !wget -O vhrdata.rar "https://drive.usercontent.google.com/download?id=1GX656JqqOyBi_Ef0w65kDGVto-nHrNs9&export=download&confirm=t&uuid=6ad01a3d-5864-4f0c-ae7e-642a14a3d197"
    !apt-get install unrar -q
    !unrar x vhrdata.rar {DATA_DIR}/
    print('Dataset downloaded and extracted.')
else:
    print('Dataset already exists.')

print('Contents:', os.listdir(DATA_DIR))

## 3. Dataset and DataLoader

The Change Detection Dataset structure:
```
vhrdata/ChangeDetectionDataset/
  Real/
    subset/
      train/
        A/  (pre-change images)
        B/  (post-change images)
        label/  (binary change masks)
      val/
        A/
        B/
        label/
```

In [ ]:
class ChangeDetectionDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = os.path.join(root_dir, 'Real', 'subset', split)
        self.transform = transform
        self.files = []

        a_dir = os.path.join(self.root_dir, 'A')
        b_dir = os.path.join(self.root_dir, 'B')
        label_dir = os.path.join(self.root_dir, 'label')

        for fname in sorted(os.listdir(a_dir)):
            if fname.endswith(('.jpg', '.png')):
                self.files.append((
                    os.path.join(a_dir, fname),
                    os.path.join(b_dir, fname),
                    os.path.join(label_dir, fname.replace('.jpg', '.png'))
                ))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        a_path, b_path, label_path = self.files[idx]

        img_a = Image.open(a_path).convert('RGB')
        img_b = Image.open(b_path).convert('RGB')
        label = Image.open(label_path).convert('L')

        if self.transform:
            img_a = self.transform(img_a)
            img_b = self.transform(img_b)
            label = self.transform(label)

        x = torch.cat([img_a, img_b], dim=0)
        y = (label > 0).float()
        return x, y


IMG_SIZE = 256
BATCH_SIZE = 8

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_dataset = ChangeDetectionDataset(DATA_DIR, split='train', transform=transform)
val_dataset = ChangeDetectionDataset(DATA_DIR, split='val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')

### Visualize a Sample

In [ ]:
def denormalize(tensor):
    return tensor.permute(1, 2, 0).numpy()

x, y = train_dataset[0]
img_a, img_b = x[:3], x[3:]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(denormalize(img_a))
axes[0].set_title('Pre-change (A)')
axes[1].imshow(denormalize(img_b))
axes[1].set_title('Post-change (B)')
axes[2].imshow(y.squeeze(), cmap='gray')
axes[2].set_title('Change Label')
plt.tight_layout()
plt.show()

## 4. UNet++ Model Definition

UNet++ replaces the plain skip connections of U-Net with **nested dense skip pathways**. Each node receives the output of all preceding nodes at the same level via concatenation, then applies convolution + activation.

### Architecture Details
- Encoder levels (0-3): DoubleConv blocks with MaxPooling
- Bottleneck (level 4): Deepest features
- Decoder: Each node `X[i][j]` (i=level, j=depth) is computed as:
  - Upsample from `X[i+1][j-1]`
  - Concatenate with all `X[i][k]` for k < j (previous nodes at same level)
  - Apply DoubleConv
- Final output: 1x1 conv + sigmoid

### Layer Dimensions
| Level | Input Channels | Output Channels | Spatial Size |
|-------|---------------|-----------------|--------------|
| 0     | 6             | 32              | 256x256      |
| 1     | 32            | 64              | 128x128      |
| 2     | 64            | 128             | 64x64        |
| 3     | 128           | 256             | 32x32        |
| 4     | 256           | 512             | 16x16        |

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class NestedUNet(nn.Module):
    def __init__(self, in_ch=6, out_ch=1, base_filters=32, deep_supervision=False):
        super().__init__()
        self.deep_supervision = deep_supervision
        n1 = base_filters
        filters = [n1, n1 * 2, n1 * 4, n1 * 8, n1 * 16]

        self.pool = nn.MaxPool2d(2, 2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.conv0_0 = DoubleConv(in_ch, filters[0])
        self.conv1_0 = DoubleConv(filters[0], filters[1])
        self.conv2_0 = DoubleConv(filters[1], filters[2])
        self.conv3_0 = DoubleConv(filters[2], filters[3])
        self.conv4_0 = DoubleConv(filters[3], filters[4])

        self.conv0_1 = DoubleConv(filters[0] + filters[1], filters[0])
        self.conv1_1 = DoubleConv(filters[1] + filters[2], filters[1])
        self.conv2_1 = DoubleConv(filters[2] + filters[3], filters[2])
        self.conv3_1 = DoubleConv(filters[3] + filters[4], filters[3])

        self.conv0_2 = DoubleConv(filters[0] * 2 + filters[1], filters[0])
        self.conv1_2 = DoubleConv(filters[1] * 2 + filters[2], filters[1])
        self.conv2_2 = DoubleConv(filters[2] * 2 + filters[3], filters[2])

        self.conv0_3 = DoubleConv(filters[0] * 3 + filters[1], filters[0])
        self.conv1_3 = DoubleConv(filters[1] * 3 + filters[2], filters[1])

        self.conv0_4 = DoubleConv(filters[0] * 4 + filters[1], filters[0])

        self.final = nn.Conv2d(filters[0], out_ch, 1)

    def forward(self, x):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0))
        x0_1 = self.conv0_1(torch.cat([x0_0, self.up(x1_0)], 1))

        x2_0 = self.conv2_0(self.pool(x1_0))
        x1_1 = self.conv1_1(torch.cat([x1_0, self.up(x2_0)], 1))
        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self.up(x1_1)], 1))

        x3_0 = self.conv3_0(self.pool(x2_0))
        x2_1 = self.conv2_1(torch.cat([x2_0, self.up(x3_0)], 1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self.up(x2_1)], 1))
        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self.up(x1_2)], 1))

        x4_0 = self.conv4_0(self.pool(x3_0))
        x3_1 = self.conv3_1(torch.cat([x3_0, self.up(x4_0)], 1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self.up(x3_1)], 1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self.up(x2_2)], 1))
        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self.up(x1_3)], 1))

        output = self.final(x0_4)
        return torch.sigmoid(output)


model = NestedUNet(in_ch=6, out_ch=1, base_filters=32).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(f'Model too large for local GPU - designed for Colab T4/P100')

## 5. Loss Function: BCE + Dice Loss

Combines Binary Cross-Entropy (pixel-level) with Dice Loss (region-level overlap) for robust change detection.

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, weight_bce=0.5, weight_dice=0.5):
        super().__init__()
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.bce = nn.BCELoss()

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)

        smooth = 1.0
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        dice_loss = 1 - (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

        return self.weight_bce * bce_loss + self.weight_dice * dice_loss


criterion = BCEDiceLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

## 6. Training Loop

Training configuration:
- Epochs: 50 (adjustable)
- Batch size: 8 (limited by GPU memory)
- Learning rate: 1e-4
- Metrics tracked: Loss, IoU, F1 Score

In [ ]:
def iou_score(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    intersection = (pred_bin * target).sum()
    union = pred_bin.sum() + target.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)


def f1_score(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    tp = (pred_bin * target).sum()
    fp = pred_bin.sum() - tp
    fn = target.sum() - tp
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    return 2 * precision * recall / (precision + recall + 1e-6)


EPOCHS = 50
train_losses, val_losses = [], []
val_ious, val_f1s = [], []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    model.eval()
    val_loss = 0
    epoch_iou, epoch_f1 = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            val_loss += criterion(pred, y).item()
            epoch_iou += iou_score(pred, y).item()
            epoch_f1 += f1_score(pred, y).item()

    val_losses.append(val_loss / len(val_loader))
    val_ious.append(epoch_iou / len(val_loader))
    val_f1s.append(epoch_f1 / len(val_loader))

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | IoU: {val_ious[-1]:.4f} | F1: {val_f1s[-1]:.4f}')

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(val_ious, label='IoU', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(val_f1s, label='F1 Score', color='red')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 8. Qualitative Results

In [ ]:
model.eval()
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

with torch.no_grad():
    for i, (x, y) in enumerate(val_loader):
        if i >= 4:
            break
        x, y = x.to(device), y.to(device)
        pred = model(x)

        for j in range(min(4, x.size(0))):
            idx = i * 4 + j
            row = idx // 4
            col = idx % 4

            img_a = x[j, :3].cpu().permute(1, 2, 0).numpy()
            gt = y[j, 0].cpu().numpy()
            pr = pred[j, 0].cpu().numpy()

            axes[row, col].imshow(img_a)
            axes[row, col].imshow(gt, alpha=0.5, cmap='Reds')
            axes[row, col].imshow(pr > 0.5, alpha=0.3, cmap='Blues')
            axes[row, col].set_title(f'GT (red) / Pred (blue)')
            axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
torch.save(model.state_dict(), 'unetpp_change_detection.pth')
print('Model saved to unetpp_change_detection.pth')